# MDE и размер выборки

Взято из раздела **MDE** лекции [Stat power.ipynb](https://github.com/avito-tech/applied_statistics/blob/main/Stat%20power.ipynb).

MDE — минимальный эффект, который можно обнаружить при заданной мощности. Например, результат `0.10` означает **+10 процентных пунктов**, а не +10% относительно исходной доли.

**Как пользоваться:** меняй параметры в блоке **ВВОД ДАННЫХ**, затем **Run All**. Графики — необязательны.


In [ ]:
from scipy.stats import binom
import numpy as numpy
from matplotlib import pyplot


## 1. ВВОД ДАННЫХ


In [ ]:
N = 30                # размер текущей выборки
MU_H0 = 0.50          # базовая вероятность успеха (50%)
ALPHA = 0.05          # уровень значимости
MIN_POWER = 0.80      # желаемая мощность (80%)
DESIRED_MDE = 0.10   # какой эффект хотим обнаружить (+10 п.п.)
MAX_N = 30000         # максимальное число наблюдений при поиске размера выборки
SHOW_PLOTS = True     # False — только числа


## 2. Функции из лекции

Функция мощности используется и в подборе MDE; исходные комментарии сохранены. Добавлена только понятная ошибка, когда подобрать MDE невозможно.


In [ ]:
def get_stat_power(N, mu_h0, mu_alternative, alpha):
    '''Вычисляет статистическую мощность критерия для биномиального распределения
    
    Параметры:
        N - количество бернуллиевских экспериментов (размер выборки)
        mu_h0 - вероятность успеха в нулевой гипотезе
        mu_alternative - предполагаемая вероятность успеха в эксперименте
        alpha - уровень значимости критерия
    '''
    binom_h0 = binom(n=N, p=mu_h0)
    binom_alternative = binom(n=N, p=mu_alternative)

    # вычисляем критическое значение
    critical_value = binom_h0.ppf(1 - alpha) + 1
    # вычисляем мощность по формуле
    return 1 - binom_alternative.cdf(critical_value - 1) 

In [ ]:
def binom_test_mde_one_sided(N, mu0, alpha=0.05, min_power=0.8):
    '''Вычисляет MDE одностороннего критерия для проверки гипотезы mu = mu0 в задаче с доставкой
    
    Параметры:
        N (int) - размер выборки
        mu0 (float) - вероятность успеха в нулевой гипотезе
        alpha (float) - уровень значимости критерия
        min_power (float) - желаемая мощность
        
    Возвращает:
        float - MDE'''
    delta_grid = numpy.linspace(0, 1 - mu0, 500) 
    power = get_stat_power(N, mu0, mu0 + delta_grid, alpha=alpha)
    # выберем подходящие delta и вернем первую
    fit_delta = delta_grid[power >= min_power]
    if fit_delta.size == 0:
        raise ValueError("При заданных параметрах целевая мощность недостижима")
    return float(fit_delta[0])

## 3. Подставляем значения и получаем результат


In [ ]:
mde = binom_test_mde_one_sided(N, MU_H0, alpha=ALPHA, min_power=MIN_POWER)
print(f"MDE для n={N}: +{mde*100:.2f} процентных пункта")

# Обычно MDE считают не просто так, а неразрывно с ним идёт вопрос об определении размера выборки.
n_grid = numpy.arange(1, MAX_N + 1)
power_by_n = get_stat_power(n_grid, MU_H0, MU_H0 + DESIRED_MDE, alpha=ALPHA)
suitable_n = n_grid[power_by_n >= MIN_POWER]
if suitable_n.size == 0:
    print("Для выбранного MDE необходимый размер выборки больше MAX_N")
else:
    min_n = int(suitable_n.min())
    print(f"Нужный размер выборки для эффекта +{DESIRED_MDE*100:.2f} п.п.: {min_n}")


## 4. График MDE: мощность при разных эффектах


In [ ]:
if SHOW_PLOTS:
    delta_grid = numpy.linspace(0, 1 - MU_H0, 500)
    power_by_effect = get_stat_power(N, MU_H0, MU_H0 + delta_grid, alpha=ALPHA)

    pyplot.figure(figsize=(10, 5))
    pyplot.plot(delta_grid * 100, power_by_effect, label="Мощность")
    pyplot.axhline(MIN_POWER, ls="--", label=f"Цель: {MIN_POWER:.0%}")
    # считаем минимальный эффект мощности в 80%
    pyplot.axvline(mde * 100, ls=":", label=f"MDE: +{mde*100:.2f} п.п.")
    pyplot.xlabel("Эффект, процентные пункты")
    pyplot.ylabel("Мощность")
    pyplot.ylim(0, 1.02)
    pyplot.title(f"Мощность от эффекта при n={N}")
    pyplot.legend()
    pyplot.show()


## 5. График: мощность при разных размерах выборки


In [ ]:
if SHOW_PLOTS:
    # можем вычислить мощность сразу для всех n
    graph_n = numpy.arange(10, min(MAX_N, max(600, N*20)) + 1, 10)
    graph_power = get_stat_power(graph_n, MU_H0, MU_H0 + DESIRED_MDE, alpha=ALPHA)

    pyplot.figure(figsize=(10, 5))
    pyplot.plot(graph_n, graph_power, label="Мощность")
    pyplot.axhline(MIN_POWER, ls="--", label=f"Цель: {MIN_POWER:.0%}")
    if suitable_n.size:
        pyplot.axvline(min_n, ls=":", label=f"Нужно n = {min_n}")
    pyplot.xlabel("Размер выборки")
    pyplot.ylabel("Мощность")
    pyplot.ylim(0, 1.02)
    pyplot.title("Подбор размера выборки")
    pyplot.legend()
    pyplot.show()
